# CareLoop Clinics — No-Show Guard
## Part 4: GenAI-assisted follow-up note analysis

Prerequisites:
- `followup_notes.csv` in `data/`
- `appointments_clean.csv` generated by Part 1
- `GEMINI_API_KEY` in a local `.env` file (see `.env.example`), or typed at the prompt in Colab

Never hard-code or save the API key in this notebook. The key is read from the environment and then deleted from the session. Before committing, clear any cell output that might contain it and search the notebook for the key's first characters.

In [ ]:
!pip install -q google-genai pydantic python-dotenv

import json
import os
import sys
import time
import pandas as pd
from pathlib import Path
from pydantic import BaseModel, Field, ValidationError
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv(Path(".env"))
API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
if not API_KEY and sys.stdin.isatty():
    API_KEY = input("Paste your Gemini API key: ").strip()
if not API_KEY or API_KEY.lower().startswith("your_"):
    raise RuntimeError("Set GEMINI_API_KEY in .env or the environment. Do not hard-code the key.")
client = genai.Client(api_key=API_KEY)
del API_KEY
MODEL_NAME = "gemini-2.5-flash"

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

SYSTEM_TEXT = (
    "You are an outpatient clinic operations analyst at CareLoop Clinics in India. "
    "Extract structured insights from post-appointment call-centre notes accurately and objectively."
)

def build_craft_prompt(note_text):
    return (
        "Context: A patient missed a scheduled outpatient appointment at CareLoop Clinics, and a call-centre agent logged a brief follow-up note.\n"
        "Role: Healthcare operations analyst.\n"
        "Action: Classify the primary reason for the missed visit into exactly one of these allowed root_cause labels: "
        "forgot_or_no_reminder, distance_or_transport, cost_or_fee, illness_or_emergency, work_or_schedule, other. "
        "If the note is ambiguous, unclear, or does not fit the first five categories, choose 'other' rather than guessing.\n"
        "Tone: Concise, neutral, and operational.\n"
        "Format: Return valid JSON with root_cause, confidence (0.0 to 1.0), summary (max 25 words), and recommended_action.\n\n"
        f"Call-centre note: \"{note_text}\""
    )

api_call_count = 0

def call_gemini_raw(prompt_text):
    global api_call_count
    api_call_count += 1
    config_kwargs = dict(
        system_instruction=SYSTEM_TEXT,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type="application/json",
        response_schema=SCHEMA,
    )
    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_text,
            config=types.GenerateContentConfig(
                **config_kwargs,
                thinking_config=types.ThinkingConfig(thinking_level="minimal")
            ),
        )
    except Exception:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_text,
            config=types.GenerateContentConfig(**config_kwargs),
        )
    return response.text

# Task 4.1
def analyse_note(note_text):
    raw_text = call_gemini_raw(build_craft_prompt(note_text))
    return json.loads(raw_text)

notes_df = pd.read_csv("data/followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]
n001_result = analyse_note(n001_text)
print("Task 4.1 (N001):", json.dumps(n001_result, indent=2))

# Task 4.2
class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

def process_first_10_notes(notes_df, out_csv="notes_analysed.csv"):
    results = []
    subset = notes_df.head(10)

    for idx, row in subset.iterrows():
        note_id = row["note_id"]
        appt_id = row["appointment_id"]
        note_text = row["note_text"]

        if pd.isna(note_text) or str(note_text).strip() == "":
            results.append({
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": "other",
                "confidence": 0.0,
                "summary": "No note recorded",
                "recommended_action": "Re-call patient",
                "validation_status": "skipped_blank",
                "raw_response": ""
            })
            continue

        raw_reply = ""
        try:
            raw_reply = call_gemini_raw(build_craft_prompt(str(note_text)))
            parsed = json.loads(raw_reply)
            insight = FollowupInsight(**parsed)
            is_allowed = insight.root_cause in ALLOWED_CAUSES
            status = "valid" if is_allowed else "invalid"

            results.append({
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": parsed.get("root_cause", ""),
                "confidence": parsed.get("confidence", ""),
                "summary": parsed.get("summary", ""),
                "recommended_action": parsed.get("recommended_action", ""),
                "validation_status": status,
                "raw_response": raw_reply
            })
        except Exception:
            parsed_partial = {}
            try:
                parsed_partial = json.loads(raw_reply) if raw_reply else {}
            except Exception:
                pass
            results.append({
                "note_id": note_id,
                "appointment_id": appt_id,
                "root_cause": parsed_partial.get("root_cause", ""),
                "confidence": parsed_partial.get("confidence", ""),
                "summary": parsed_partial.get("summary", ""),
                "recommended_action": parsed_partial.get("recommended_action", ""),
                "validation_status": "invalid",
                "raw_response": raw_reply
            })

        time.sleep(13)

    analysed_df = pd.DataFrame(results)
    analysed_df.to_csv(out_csv, index=False)
    print(f"\nTask 4.2 - Saved {out_csv} ({len(analysed_df)} rows):")
    print(analysed_df[["note_id", "appointment_id", "root_cause", "confidence", "validation_status"]].to_string(index=False))
    return analysed_df

analysed_df = process_first_10_notes(notes_df)

# Task 4.3
print("\n--- Task 4.3 (a): Deliberate Pydantic Validation Errors ---")
bad_cases = [
    {"root_cause": "other", "confidence": 1.7, "summary": "Patient missed visit", "recommended_action": "Call back"},
    {"root_cause": "other", "confidence": 0.8, "summary": "Patient missed visit"}
]
for i, bad in enumerate(bad_cases, 1):
    try:
        FollowupInsight(**bad)
    except ValidationError as e:
        print(f"Bad case {i} caught:\n{e}\n")

print("--- Task 4.3 (b): V1 vs V2 Prompt Comparison on N007 & N009 ---")
comparison_rows = []
for nid in ["N007", "N009"]:
    ntext = notes_df.loc[notes_df["note_id"] == nid, "note_text"].iloc[0]

    raw_v1 = call_gemini_raw(f"Classify this note: {ntext}")
    time.sleep(13)
    d_v1 = json.loads(raw_v1)
    rc_v1 = d_v1.get("root_cause", "")
    comparison_rows.append({
        "note_id": nid, "prompt": "V1 (Minimal)",
        "root_cause": rc_v1, "in_allowed_list": rc_v1 in ALLOWED_CAUSES
    })

    raw_v2 = call_gemini_raw(build_craft_prompt(ntext))
    time.sleep(13)
    d_v2 = json.loads(raw_v2)
    rc_v2 = d_v2.get("root_cause", "")
    comparison_rows.append({
        "note_id": nid, "prompt": "V2 (CRAFT)",
        "root_cause": rc_v2, "in_allowed_list": rc_v2 in ALLOWED_CAUSES
    })

print(pd.DataFrame(comparison_rows).to_string(index=False))
print(f"\nTask 4.3 (c) - Total Gemini API calls used across Tasks 4.1–4.3: {api_call_count}")

# Task 4.4
clean_appts = pd.read_csv("appointments_clean.csv")
merged_notes = analysed_df.merge(clean_appts, on="appointment_id", how="inner")
ct = pd.crosstab(merged_notes["root_cause"], merged_notes["reminder_sent"])
print("\n--- Task 4.4 (a): Cross-tab of root_cause vs reminder_sent ---")
print(ct)